# 04 価格説明モデル プロトタイプ

公示地価の単価（円/m²）を目的変数に、
施設特徴量・地形特徴量・地域属性を説明変数として価格説明モデルを構築する。

## 前提
- `location_features` テーブルに施設・地形特徴量が保存済みであること
  - `recompute_public_notice_features.py --pref 13 --year 2026` を先に実行する
- `land_prices_public_notice` テーブルに公示地価データがあること

## 実施内容
1. データロード・結合
2. 特徴量エンジニアリング
3. 線形回帰ベースライン
4. LightGBM モデル
5. 特徴量重要度・残差分析
6. 物件の相対的な割安/割高推定への接続

In [ ]:
import sys
from pathlib import Path

# プロジェクトルートを sys.path に追加
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'IPAexGothic'  # 日本語フォント（未インストールの場合は変更）

import db

warnings.filterwarnings('ignore')
print('準備完了')

## 1. データロード・結合

In [ ]:
conn = db.get_connection(read_only=True)

# 公示地価（用途・都道府県・価格・座標）
land = conn.execute("""
    SELECT
        p.point_id,
        p.year,
        p.prefecture_code,
        p.prefecture_name,
        p.city_code,
        p.city_name,
        p.use_category_code,
        p.use_category_name,
        p.price_yen_per_sqm,
        p.yoy_change_pct,
        p.area_sqm,
        p.building_coverage_ratio,
        p.floor_area_ratio,
        p.lat,
        p.lon
    FROM land_prices_public_notice p
    WHERE p.price_yen_per_sqm IS NOT NULL
      AND p.lat IS NOT NULL
      AND p.lon IS NOT NULL
""").fetchdf()

print(f'公示地価: {len(land):,} 件')
print(f'年度: {sorted(land["year"].unique())}')
land.head(3)

In [ ]:
# location_features（施設・地形特徴量）
feats = conn.execute("""
    SELECT lf.*
    FROM location_features lf
    INNER JOIN public_notice_location_features pnlf
        ON lf.location_key = pnlf.location_key
""").fetchdf()

print(f'特徴量キャッシュ: {len(feats):,} 件')
feats.head(3)

In [ ]:
# public_notice_location_features でリンク
links = conn.execute("""
    SELECT point_id, year, location_key
    FROM public_notice_location_features
""").fetchdf()

# 結合
df = land.merge(links, on=['point_id', 'year'], how='inner')
df = df.merge(feats.drop(columns=['lat','lon','city_code'], errors='ignore'), on='location_key', how='left')

print(f'結合後: {len(df):,} 件（特徴量あり: {df["convenience_count_1000m"].notna().sum():,} 件）')
df.head(3)

## 2. 特徴量エンジニアリング

In [ ]:
# 目的変数: log 変換（価格は右裾が長い分布）
df['log_price'] = np.log1p(df['price_yen_per_sqm'])

# 用途コードをカテゴリ化
df['use_cat'] = df['use_category_code'].fillna('unknown').astype('category')

# 生活利便スコア: コンビニ+スーパー最寄り距離の逆数（近いほど高スコア）
df['convenience_score'] = (
    1000 / (df['convenience_nearest_m'].clip(lower=50).fillna(1000)) +
    1000 / (df['supermarket_nearest_m'].clip(lower=50).fillna(1000))
)

# 交通アクセススコア: 駅+バス停最寄り距離の逆数
df['transit_score'] = (
    1000 / (df['transit_nearest_m'].clip(lower=50).fillna(2000))
)

# 水害リスクスコア: 標高が低く水辺が近いほど高リスク
df['flood_risk_score'] = (
    (10 - df['elevation_m'].clip(0, 50).fillna(10)) / 10 +
    1000 / (df['nearest_water_m'].clip(lower=50).fillna(1000))
).clip(0, 3)

# 容積率（欠損を中央値で埋め）
far_median = df['floor_area_ratio'].median()
df['far_filled'] = df['floor_area_ratio'].fillna(far_median)

print('特徴量作成完了')
df[['log_price','convenience_score','transit_score','flood_risk_score','far_filled']].describe()

In [ ]:
# 説明変数リスト
FEATURE_COLS = [
    # 施設系
    'convenience_count_1000m',
    'supermarket_count_1000m',
    'transit_count_1000m',
    'school_count_1000m',
    'medical_count_1000m',
    'park_count_1000m',
    'pachinko_count_1000m',
    'convenience_score',
    'transit_score',
    # 地形系
    'elevation_m',
    'nearest_water_m',
    'water_count_1000m',
    'flood_risk_score',
    # 物件属性
    'far_filled',
    'building_coverage_ratio',
    'area_sqm',
]
TARGET_COL = 'log_price'

# 学習データ: 特徴量が揃っている行のみ
df_model = df[FEATURE_COLS + [TARGET_COL, 'use_cat', 'prefecture_code', 'year']].dropna(subset=['convenience_count_1000m', 'elevation_m', TARGET_COL])
print(f'モデル用データ: {len(df_model):,} 件')

# 都道府県・用途コードをラベルエンコード
from sklearn.preprocessing import LabelEncoder
le_pref = LabelEncoder()
le_use  = LabelEncoder()
df_model = df_model.copy()
df_model['pref_enc'] = le_pref.fit_transform(df_model['prefecture_code'])
df_model['use_enc']  = le_use.fit_transform(df_model['use_cat'].astype(str))

X_COLS = FEATURE_COLS + ['pref_enc', 'use_enc', 'year']
X = df_model[X_COLS].fillna(0).values
y = df_model[TARGET_COL].values

## 3. 線形回帰ベースライン

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0)),
])

scores = cross_val_score(lr_pipe, X, y, cv=5, scoring='r2')
print(f'線形回帰 R² (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}')

lr_pipe.fit(X, y)
y_pred_lr = lr_pipe.predict(X)

plt.figure(figsize=(5,4))
plt.scatter(y, y_pred_lr, alpha=0.3, s=5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=1)
plt.xlabel('実際の log(価格)')
plt.ylabel('予測 log(価格)')
plt.title('線形回帰: 実測 vs 予測')
plt.tight_layout()
plt.show()

## 4. LightGBM モデル

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print('LightGBM 未インストール。pip install lightgbm でインストールしてください。')
    HAS_LGB = False

if HAS_LGB:
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    lgb_model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=0)],
    )

    from sklearn.metrics import r2_score, mean_absolute_error
    y_pred_lgb = lgb_model.predict(X_test)
    r2 = r2_score(y_test, y_pred_lgb)
    mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_lgb))
    print(f'LightGBM テスト R²: {r2:.3f}')
    print(f'LightGBM テスト MAE (実価格): {mae:,.0f} 円/m²')

## 5. 特徴量重要度・残差分析

In [ ]:
if HAS_LGB:
    importance = pd.Series(lgb_model.feature_importances_, index=X_COLS).sort_values(ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # 特徴量重要度
    importance.plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('特徴量重要度 (LightGBM)')
    axes[0].set_xlabel('重要度')

    # 残差プロット
    residuals = y_test - y_pred_lgb
    axes[1].scatter(y_pred_lgb, residuals, alpha=0.3, s=5)
    axes[1].axhline(0, color='red', linestyle='--', lw=1)
    axes[1].set_xlabel('予測 log(価格)')
    axes[1].set_ylabel('残差')
    axes[1].set_title('残差プロット')

    plt.tight_layout()
    plt.show()

## 6. 施設特徴量の価格への影響（SHAP）

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    print('shap 未インストール。pip install shap でインストールしてください。')
    HAS_SHAP = False

if HAS_LGB and HAS_SHAP:
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X_test[:500])  # 500件でサンプリング

    plt.figure(figsize=(8, 5))
    shap.summary_plot(shap_values, X_test[:500], feature_names=X_COLS, show=False)
    plt.title('SHAP 特徴量重要度（価格への影響）')
    plt.tight_layout()
    plt.show()

## 7. 個別物件の割安/割高推定

掲載物件（listing_master）の特徴量を使って、
モデルの予測価格と掲載価格の乖離率を計算する。

In [ ]:
if HAS_LGB:
    # listing_master と location_features を結合
    listings = conn.execute("""
        SELECT
            lm.listing_id,
            lm.property_name,
            lm.asking_price_yen,
            lm.building_area_sqm,
            lm.land_area_sqm,
            lm.lat,
            lm.lon,
            lf.*
        FROM listing_master lm
        LEFT JOIN location_features lf ON lm.location_key = lf.location_key
        WHERE lm.asking_price_yen IS NOT NULL
          AND lm.land_area_sqm IS NOT NULL
          AND lm.land_area_sqm > 0
    """).fetchdf()

    print(f'掲載物件: {len(listings):,} 件')

    if not listings.empty:
        listings['convenience_score'] = (
            1000 / (listings['convenience_nearest_m'].clip(lower=50).fillna(1000)) +
            1000 / (listings['supermarket_nearest_m'].clip(lower=50).fillna(1000))
        )
        listings['transit_score'] = 1000 / (listings['transit_nearest_m'].clip(lower=50).fillna(2000))
        listings['flood_risk_score'] = (
            (10 - listings['elevation_m'].clip(0, 50).fillna(10)) / 10 +
            1000 / (listings['nearest_water_m'].clip(lower=50).fillna(1000))
        ).clip(0, 3)
        listings['far_filled'] = listings['floor_area_ratio'].fillna(far_median)
        listings['pref_enc'] = 0  # 未知都道府県は 0
        listings['use_enc'] = 0
        listings['year'] = 2026

        X_lst = listings[X_COLS].fillna(0).values
        y_pred_lst = np.expm1(lgb_model.predict(X_lst))

        listings['model_price_yen_per_sqm'] = y_pred_lst
        listings['asking_price_yen_per_sqm'] = listings['asking_price_yen'] / listings['land_area_sqm']
        listings['model_gap_pct'] = (
            (listings['asking_price_yen_per_sqm'] - listings['model_price_yen_per_sqm'])
            / listings['model_price_yen_per_sqm'] * 100
        )

        result = listings[['property_name', 'asking_price_yen_per_sqm', 'model_price_yen_per_sqm', 'model_gap_pct']].copy()
        result['asking_price_yen_per_sqm'] = result['asking_price_yen_per_sqm'].map(lambda x: f'{x:,.0f}')
        result['model_price_yen_per_sqm']  = result['model_price_yen_per_sqm'].map(lambda x: f'{x:,.0f}')
        result['model_gap_pct']            = result['model_gap_pct'].map(lambda x: f'{x:+.1f}%')
        result.columns = ['物件名', '掲載単価(円/m²)', 'モデル単価(円/m²)', 'モデル乖離率']
        display(result.head(20))
    else:
        print('掲載物件データなし。物件分析タブで物件を分析してから再実行してください。')